# Đánh giá mô hình (Model Evaluation)

Notebook này sẽ tự động tải các tham số trọng số từ 3 mô hình `.pt` (`GCN`, `GAT`, `GIN`), thực hiện dự đoán trên tập Valid và tính toán các độ đo bao gồm: `Accuracy`, `Precision`, `Recall`, và `F1-Score`.

*Lưu ý: Tập Test mặc định của Project này không có nhãn (nhãn = 2 ~ Unknown), nên chúng ta sử dụng dữ liệu đã gán nhãn ở tập valid để đo đạc lường các độ đo này.*

In [1]:
import torch
import pandas as pd
from omegaconf import OmegaConf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from fraud_detection import Trainer, GCN, GAT, GIN, EllipticDataset

## 1. Vòng lặp lấy chỉ số từ 3 model
Tất cả các metrics này sẽ được lưu lại báo cáo thành dạng bảng qua thư viện `pandas`.

In [2]:
models_to_evaluate = ["gcn", "gat", "gin"]
results = []

for model_name in models_to_evaluate:
    config_path = f"configs/elliptic_{model_name}.yaml"
    weight_path = f"weights/elliptic_{model_name}.pt"
    
    # 1. Đọc config từ file yaml tương ứng
    config = OmegaConf.load(config_path)
    
    # 2. Khởi tạo Dataset để lấy thông tin các thuộc tính về Node
    dataset = EllipticDataset(config.dataset)
    config.model.input_dim = dataset.pyg_dataset().num_node_features
    
    # 3. Khởi tạo Object mô hình (GCN/GAT/GIN)
    if model_name == "gcn":
        model = GCN(config.model)
    elif model_name == "gat":
        model = GAT(config.model)
    elif model_name == "gin":
        model = GIN(config.model)
        
    # 4. Tải tham số model (trọng số file .pt)
    model.load_state_dict(torch.load(weight_path))
    
    # 5. Phục hồi môi trường với object Trainer
    trainer = Trainer(config)
    trainer.model = model.double().to(config.train.device)
    
    # 6. Dự đoán bằng cách dùng labeled_only=True để lấy hết nhãn sau đó cắt theo Valid_idx.
    # Lưu ý: Không dùng test_idx vì test_idx chỉ bao gồm các node chưa biết nhãn (Unknown labels/class 2)
    preds, _ = trainer.test(threshold=0.5, labeled_only=True)
    
    valid_idx = trainer.dataset.valid_idx
    pred_probs = preds[valid_idx]
    pred_labels = pred_probs > 0.5
    
    # Ground truth tương ứng
    true_labels = trainer.dataset.y.detach().cpu().numpy()[valid_idx]
    
    # 7. Tính độ đo bằng sklearn
    acc = accuracy_score(true_labels, pred_labels)
    prec = precision_score(true_labels, pred_labels, zero_division=0)
    rec = recall_score(true_labels, pred_labels, zero_division=0)
    
    # Khi tính F1-Score, nếu target chỉ có 0-1, mặc định average là 'binary' (không dùng macro/micro trừ khi multiclass)
    f1 = f1_score(true_labels, pred_labels, zero_division=0)
    
    # Cache kết quả
    results.append({
        "Model": model_name.upper(),
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1
    })

df_results = pd.DataFrame(results)
display(df_results)

,Model,Accuracy,Precision,Recall,F1-Score
0,GCN,0.937852,0.525188,0.452447,0.486111
1,GAT,0.953929,0.746479,0.440443,0.554007
2,GIN,0.947211,0.662400,0.382271,0.484778
